# Hello world — IBM Quantum (laboratorio)

Basado en el tutorial oficial: https://quantum.cloud.ibm.com/docs/en/ibm-quantum-compute/hello-world

Se siguen los 4 pasos de **Qiskit patterns**: (1) mapear el problema a circuitos y observables, (2) optimizar para el hardware, (3) ejecutar con un primitivo, (4) post-procesar.

* `USAR_HARDWARE = False` → simulador local con ruido (no necesita cuenta).
* `USAR_HARDWARE = True` → computadora cuántica real de IBM. Antes ejecuta **una vez** `python 01_guardar_cuenta.py` para guardar tu API key.

In [ ]:
# Cambia a True para ejecutar en una computadora cuantica real de IBM
USAR_HARDWARE = False

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.options_models import EstimatorOptions
from qiskit_ibm_runtime.executor_estimator import Estimator
from qiskit_ibm_runtime.fake_provider import FakeBelemV2, FakeGuadalupeV2
import matplotlib.pyplot as plt

if USAR_HARDWARE:
    service = QiskitRuntimeService()

## Parte 1: Estado de Bell

### Paso 1 — Mapear el problema a un circuito cuántico y observables

Compuerta Hadamard en el qubit 0 y CNOT del qubit 0 al 1 → estado de Bell $\frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$.

In [ ]:
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.draw("mpl")

Seis observables de Pauli de 2 qubits. Para el estado de Bell se espera $\langle ZZ\rangle = \langle XX\rangle = 1$ y los de un solo qubit ≈ 0.

In [ ]:
observables_labels = ["IZ", "IX", "ZI", "XI", "ZZ", "XX"]
observables = [SparsePauliOp(label) for label in observables_labels]

### Paso 2 — Optimizar el circuito para el hardware (transpilar a ISA)

In [ ]:
if USAR_HARDWARE:
    backend = service.least_busy(simulator=False, operational=True)
else:
    backend = FakeBelemV2()
print("Backend:", backend.name)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
isa_circuit.draw("mpl", idle_wires=False)

In [ ]:
mapped_observables = [observable.apply_layout(isa_circuit.layout) for observable in observables]

### Paso 3 — Ejecutar con el Estimator

In [ ]:
estimator = Estimator(mode=backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 5000

job = estimator.run([(isa_circuit, mapped_observables)])
print(f">>> Job ID: {job.job_id()}")

### Paso 4 — Analizar los resultados

`job.result()` espera hasta que el job termine (en hardware puede haber cola).

In [ ]:
pub_result = job.result()[0]
values = pub_result.data.evs
errors = pub_result.data.stds

for lbl, v, e in zip(observables_labels, values, errors):
    print(f"{lbl:<4} {v:>7.4f} +/- {e:.4f}")

plt.plot(observables_labels, values, "-o")
plt.xlabel("Observables")
plt.ylabel("Values")
plt.show()

## Parte 2: Escalar a un estado GHZ de 100 qubits

En hardware se usan **n = 100** qubits como en el tutorial. En el simulador local se usan **n = 10** (100 qubits no se pueden simular en una PC).

### Paso 1 — Mapear el problema

In [ ]:
def get_qc_for_n_qubit_GHZ_state(n: int) -> QuantumCircuit:
    """Devuelve un QuantumCircuit que prepara el estado GHZ de n qubits."""
    if isinstance(n, int) and n >= 2:
        qc = QuantumCircuit(n)
        qc.h(0)
        for i in range(n - 1):
            qc.cx(i, i + 1)
    else:
        raise Exception("n no es un entero valido (debe ser >= 2)")
    return qc


n = 100 if USAR_HARDWARE else 10
qc = get_qc_for_n_qubit_GHZ_state(n)

Operadores $Z_0 Z_i$: `ZZII...I`, `ZIZI...I`, …, `ZIII...Z` ($n-1$ operadores).

In [ ]:
operator_strings = ["Z" + "I" * i + "Z" + "I" * (n - 2 - i) for i in range(n - 1)]
print(operator_strings[:3], "...")
print(len(operator_strings))

operators = [SparsePauliOp(operator) for operator in operator_strings]

### Paso 2 — Optimizar para el hardware

In [ ]:
if USAR_HARDWARE:
    backend = service.least_busy(simulator=False, operational=True, min_num_qubits=n)
else:
    backend = FakeGuadalupeV2()
print("Backend:", backend.name, f"({backend.num_qubits} qubits)")

pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc)
isa_operators_list = [op.apply_layout(isa_circuit.layout) for op in operators]

### Paso 3 — Ejecutar (con supresión de errores: dynamical decoupling XY4)

In [ ]:
options = EstimatorOptions()
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"

estimator = Estimator(backend, options=options)

job = estimator.run([(isa_circuit, isa_operators_list)])
job_id = job.job_id()
print(job_id)

### Paso 4 — Post-procesar y graficar

Idealmente todos los valores normalizados serían 1; el ruido hace que la correlación decaiga con la distancia.

In [ ]:
data = list(range(1, len(operators) + 1))  # distancia entre qubits
result = job.result()[0]
values = result.data.evs
values = [v / values[0] for v in values]  # normalizar

plt.plot(data, values, marker="o", label=f"{n}-qubit GHZ state")
plt.xlabel("Distance between qubits $i$")
plt.ylabel(r"$\langle Z_i Z_0 \rangle / \langle Z_1 Z_0 \rangle $")
plt.legend()
plt.show()